# Part 7: Benchmarks (Problems 051–058)

After building the server, we need to measure how well it performs.

### Key Metrics

| Metric | What it measures | Lower is better? |
|--------|-----------------|------------------|
| **TTFT** (Time to First Token) | Latency from request arrival to first token | Yes |
| **TPOT** (Time Per Output Token) | Average time to generate each subsequent token | Yes |
| **Throughput** | Total tokens generated per second across all requests | No — higher is better |
| **P50/P95/P99 latency** | Percentile latencies for TTFT | Yes |

### Why these matter

- **TTFT** determines perceived responsiveness — users notice when the first word takes a long time
- **TPOT** determines streaming quality — high TPOT means tokens appear slowly and jerkily
- **Throughput** determines server cost — higher throughput means fewer GPUs per user

## Cell 1: Generate synthetic requests, show prompt length distribution

In [ ]:
import importlib
import random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.052_build_synthetic_request_generator")
    build_synthetic_request_generator = _m.build_synthetic_request_generator
except Exception:
    print("Solve problem 052 first:")
    print("  cp problems/052_build_synthetic_request_generator.py solutions/052_build_synthetic_request_generator.py")
    build_synthetic_request_generator = None

if build_synthetic_request_generator is not None:
    generator = build_synthetic_request_generator(seed=42)
    requests = [next(generator) for _ in range(200)]
    prompt_lengths = [r.get('prompt_tokens', len(r.get('prompt', '').split())) for r in requests]
    output_lengths = [r.get('max_tokens', r.get('output_tokens', 50)) for r in requests]
else:
    # Generate synthetic data for illustration
    random.seed(42)
    # Prompt lengths follow a log-normal distribution (many short, few long)
    prompt_lengths = [max(4, int(np.random.lognormal(3.5, 0.8))) for _ in range(200)]
    output_lengths = [max(1, int(np.random.lognormal(3.0, 0.6))) for _ in range(200)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(prompt_lengths, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Prompt Length (tokens)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Prompt Length Distribution\n(n=200, mean={np.mean(prompt_lengths):.0f}, median={np.median(prompt_lengths):.0f})")
axes[0].axvline(np.mean(prompt_lengths), color="red", linestyle="--", label=f"mean={np.mean(prompt_lengths):.0f}")
axes[0].legend()

axes[1].hist(output_lengths, bins=30, color="tomato", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Output Length (tokens)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Output Length Distribution\n(n=200, mean={np.mean(output_lengths):.0f}, median={np.median(output_lengths):.0f})")
axes[1].axvline(np.mean(output_lengths), color="darkred", linestyle="--", label=f"mean={np.mean(output_lengths):.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/request_distribution.png", dpi=100)
plt.show()

print(f"Synthetic request statistics:")
print(f"  Prompt lengths: min={min(prompt_lengths)}, max={max(prompt_lengths)}, "
      f"p50={int(np.percentile(prompt_lengths, 50))}, p95={int(np.percentile(prompt_lengths, 95))}")
print(f"  Output lengths: min={min(output_lengths)}, max={max(output_lengths)}, "
      f"p50={int(np.percentile(output_lengths, 50))}, p95={int(np.percentile(output_lengths, 95))}")

## Cell 2: Run single request benchmark — print TTFT and TPOT

In [ ]:
import importlib
import time

try:
    _m = importlib.import_module("solutions.051_define_benchmark_metrics")
    BenchmarkMetrics = _m.BenchmarkMetrics
except Exception:
    print("Solve problem 051 first:")
    print("  cp problems/051_define_benchmark_metrics.py solutions/051_define_benchmark_metrics.py")
    BenchmarkMetrics = None

try:
    _m = importlib.import_module("solutions.053_run_single_request_benchmark")
    run_single_request_benchmark = _m.run_single_request_benchmark
except Exception:
    print("Solve problem 053 first:")
    print("  cp problems/053_run_single_request_benchmark.py solutions/053_run_single_request_benchmark.py")
    run_single_request_benchmark = None

if run_single_request_benchmark is not None:
    request = {
        "prompt": "Explain the concept of attention in transformers in detail.",
        "max_tokens": 50,
        "temperature": 0.8,
    }

    print(f"Benchmarking single request:")
    print(f"  Prompt: '{request['prompt']}'")
    print(f"  max_tokens: {request['max_tokens']}")
    print()

    metrics = run_single_request_benchmark(request)

    if isinstance(metrics, dict):
        ttft = metrics.get('ttft_ms', metrics.get('time_to_first_token', 0) * 1000)
        tpot = metrics.get('tpot_ms', metrics.get('time_per_output_token', 0) * 1000)
        total = metrics.get('total_ms', metrics.get('total_time', 0) * 1000)
        tokens = metrics.get('tokens_generated', '?')
    else:
        ttft = getattr(metrics, 'ttft_ms', 0)
        tpot = getattr(metrics, 'tpot_ms', 0)
        total = getattr(metrics, 'total_ms', 0)
        tokens = getattr(metrics, 'tokens_generated', '?')

    print(f"  Results:")
    print(f"    TTFT (Time to First Token) : {ttft:.2f} ms")
    print(f"    TPOT (Time Per Output Token): {tpot:.2f} ms")
    print(f"    Total time                  : {total:.2f} ms")
    print(f"    Tokens generated            : {tokens}")
    if tpot > 0:
        print(f"    Throughput (1 request)      : {1000/tpot:.1f} tokens/sec")
else:
    print("Implement problem 053 to run the single request benchmark.")
    print()
    print("Expected output format:")
    print("  TTFT (Time to First Token) :  23.4 ms")
    print("  TPOT (Time Per Output Token):   8.2 ms")
    print("  Total time                  : 433.4 ms")
    print("  Tokens generated            : 50")
    print("  Throughput (1 request)      : 121.9 tokens/sec")

## Cell 3: Plot latency vs batch size

In [ ]:
import importlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.055_plot_latency_vs_batch_size")
    plot_latency_vs_batch_size = _m.plot_latency_vs_batch_size
except Exception:
    print("Solve problem 055 first:")
    print("  cp problems/055_plot_latency_vs_batch_size.py solutions/055_plot_latency_vs_batch_size.py")
    plot_latency_vs_batch_size = None

if plot_latency_vs_batch_size is not None:
    fig = plot_latency_vs_batch_size(batch_sizes=[1, 2, 4, 8, 16, 32])
    plt.savefig("/tmp/latency_vs_batch.png", dpi=100)
    plt.show()
else:
    # Show what the plot should look like
    batch_sizes = [1, 2, 4, 8, 16, 32]
    # Latency increases with batch size (more requests compete for GPU)
    # but not linearly — GPUs are good at parallel computation
    ttft_p50 = [20, 22, 28, 45, 85, 160]
    ttft_p95 = [35, 40, 55, 90, 165, 310]
    ttft_p99 = [55, 65, 85, 140, 260, 490]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(batch_sizes, ttft_p50, 'o-', label="p50", color="steelblue")
    ax.plot(batch_sizes, ttft_p95, 's--', label="p95", color="tomato")
    ax.plot(batch_sizes, ttft_p99, '^:', label="p99", color="purple")
    ax.fill_between(batch_sizes, ttft_p50, ttft_p99, alpha=0.1, color="steelblue")
    ax.set_xlabel("Batch Size")
    ax.set_ylabel("TTFT (ms)")
    ax.set_title("TTFT Latency vs Batch Size (expected shape)\n[solve problem 055 for real data]")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/latency_vs_batch_expected.png", dpi=100)
    plt.show()
    print("Key insight: latency increases with batch size, but throughput also increases.")
    print("There is an optimal batch size that balances latency and throughput.")

## Cell 4: Plot throughput vs concurrency

In [ ]:
import importlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.056_plot_throughput_vs_concurrency")
    plot_throughput_vs_concurrency = _m.plot_throughput_vs_concurrency
except Exception:
    print("Solve problem 056 first:")
    print("  cp problems/056_plot_throughput_vs_concurrency.py solutions/056_plot_throughput_vs_concurrency.py")
    plot_throughput_vs_concurrency = None

if plot_throughput_vs_concurrency is not None:
    fig = plot_throughput_vs_concurrency(concurrency_levels=[1, 2, 4, 8, 16, 32, 64])
    plt.savefig("/tmp/throughput_vs_concurrency.png", dpi=100)
    plt.show()
else:
    # Illustrative shape
    concurrency = [1, 2, 4, 8, 16, 32, 64]
    # Throughput rises with concurrency up to the saturation point,
    # then levels off or drops (queuing theory / Little's Law)
    throughput = [85, 165, 310, 560, 820, 880, 870]
    p95_latency = [35, 40, 55, 90, 160, 340, 680]

    fig, ax1 = plt.subplots(figsize=(9, 5))
    color_t = "steelblue"
    color_l = "tomato"

    ax1.plot(concurrency, throughput, 'o-', color=color_t, linewidth=2)
    ax1.set_xlabel("Concurrent Requests")
    ax1.set_ylabel("Throughput (tokens/sec)", color=color_t)
    ax1.tick_params(axis="y", labelcolor=color_t)

    ax2 = ax1.twinx()
    ax2.plot(concurrency, p95_latency, 's--', color=color_l, linewidth=2)
    ax2.set_ylabel("P95 TTFT (ms)", color=color_l)
    ax2.tick_params(axis="y", labelcolor=color_l)

    ax1.set_title("Throughput & Latency vs Concurrency (expected shape)\n[solve problem 056 for real data]")
    ax1.grid(True, alpha=0.3)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color=color_t, marker='o', label='Throughput (tokens/sec)'),
        Line2D([0], [0], color=color_l, marker='s', linestyle='--', label='P95 Latency (ms)'),
    ]
    ax1.legend(handles=legend_elements, loc="center right")
    plt.tight_layout()
    plt.savefig("/tmp/throughput_vs_concurrency_expected.png", dpi=100)
    plt.show()
    print()
    print("Key insight: throughput saturates around 32 concurrent requests.")
    print("Beyond that, the server is GPU-bound and latency grows without throughput gain.")

## Cell 5: Generate the full benchmark report

In [ ]:
import importlib
import datetime

try:
    _m = importlib.import_module("solutions.058_write_benchmark_report")
    write_benchmark_report = _m.write_benchmark_report
except Exception:
    print("Solve problem 058 first:")
    print("  cp problems/058_write_benchmark_report.py solutions/058_write_benchmark_report.py")
    write_benchmark_report = None

try:
    _m = importlib.import_module("solutions.054_run_concurrent_request_benchmark")
    run_concurrent_request_benchmark = _m.run_concurrent_request_benchmark
except Exception:
    run_concurrent_request_benchmark = None

if write_benchmark_report is not None and run_concurrent_request_benchmark is not None:
    print("Running concurrent benchmark...")
    bench_results = run_concurrent_request_benchmark(
        n_requests=20,
        concurrency=4,
    )
    report_path = "/tmp/benchmark_report.txt"
    write_benchmark_report(bench_results, output_path=report_path)
    with open(report_path) as f:
        print(f.read())
else:
    # Show what a complete benchmark report looks like
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    report = f"""
=====================================
 Mini LLM Inference Server — Benchmark Report
 Generated: {now}
=====================================

Configuration:
  Model         : distilgpt2
  Max batch size: 8
  Page pool size: 256
  Total requests: 100
  Concurrency   : 8

Latency (Time to First Token):
  P50  :   23.4 ms
  P90  :   41.7 ms
  P95  :   58.2 ms
  P99  :   97.1 ms
  Mean :   26.8 ms

Generation Speed:
  Avg TPOT        :  8.3 ms/token
  Throughput      : 982 tokens/sec
  Total tokens    : 4,820
  Total time      : 4.91 s

Errors:
  Timeout (>5s)   : 0
  Server errors   : 0
  Success rate    : 100.0%

Comparison:
  Paged attn speedup vs naive : 2.8x memory efficiency
  Continuous batching speedup : 3.1x throughput vs static
=====================================
"""
    print(report)
    print("[This is a sample report — implement problems 054 and 058 for real data]")